In [31]:
import pandas as pd
import numpy as np

In [32]:
data=pd.read_csv("powerplant_data.csv")

In [33]:
data.head()

,AT,V,AP,RH,PE
0,8.34,40.77,1010.84,90.01,480.48
1,23.64,58.49,1011.40,74.20,445.75
2,29.74,56.90,1007.15,41.91,438.76
3,19.07,49.69,1007.22,76.79,453.09
4,11.80,40.66,1017.13,97.20,464.43


In [34]:
#AT=temp
#V=vccume
#AP=pressure
#RH=humidity

#PE is the o/p produced energy

In [35]:
data.isnull().sum()

AT    0
V     0
AP    0
RH    0
PE    0
dtype: int64

In [36]:
x = data.drop("PE", axis=1)
y = data["PE"]


In [37]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [38]:
import torch
import torch.nn as nn

x_train_tensor=torch.tensor(x_train_scaled,dtype=torch.float32)
y_train_tensor=torch.tensor(y_train.values,dtype=torch.float32).view(-1,1)

x_test_tensor=torch.tensor(x_test_scaled,dtype=torch.float32)
y_test_tensor=torch.tensor(y_test.values,dtype=torch.float32).view(-1,1)

In [39]:
from torch.utils.data import TensorDataset, DataLoader

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

In [40]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN, self).__init__()

        self.model = nn.Sequential(
            nn.Linear(x_train.shape[1], 6),
            nn.ReLU(),
            nn.Linear(6, 6),
            nn.ReLU(),
            nn.Linear(6, 1)
        )

    def forward(self, x):
        return self.model(x)

In [41]:
import torch.optim as optim

model = ANN()
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [42]:
# Lists to store losses
training_loss = []
val_losses = []

# Number of epochs
epochs = 100

for e in range(epochs):

    # -------------------
    # Training Phase
    # -------------------
    model.train()
    running_loss = 0.0

    for xb, yb in train_loader:

        optimizer.zero_grad()

        outputs = model(xb)

        loss = criterion(outputs, yb)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_train_loss = running_loss / len(train_loader)

    training_loss.append(epoch_train_loss)

    # -------------------
    # Validation Phase
    # -------------------
    model.eval()

    running_val_loss = 0.0

    with torch.no_grad():

        for xb, yb in test_loader:

            outputs = model(xb)

            loss = criterion(outputs, yb)

            running_val_loss += loss.item()

    epoch_val_loss = running_val_loss / len(test_loader)

    val_losses.append(epoch_val_loss)

    # Print progress
    print(
        f"Epoch [{e+1}/{epochs}] | "
        f"Train Loss: {epoch_train_loss:.4f} | "
        f"Val Loss: {epoch_val_loss:.4f}"
    )

print("Training Complete!")

Epoch [1/100] | Train Loss: 206858.2249 | Val Loss: 206105.4562
Epoch [2/100] | Train Loss: 203849.7587 | Val Loss: 198958.5542
Epoch [3/100] | Train Loss: 187194.6090 | Val Loss: 171094.4906
Epoch [4/100] | Train Loss: 148578.7161 | Val Loss: 123736.6294
Epoch [5/100] | Train Loss: 97832.2159 | Val Loss: 73452.2115
Epoch [6/100] | Train Loss: 54179.5608 | Val Loss: 38802.0480
Epoch [7/100] | Train Loss: 30042.7694 | Val Loss: 23947.5122
Epoch [8/100] | Train Loss: 21071.7534 | Val Loss: 18857.4229
Epoch [9/100] | Train Loss: 17548.0985 | Val Loss: 15998.0387
Epoch [10/100] | Train Loss: 14896.6974 | Val Loss: 13437.3181
Epoch [11/100] | Train Loss: 12363.6671 | Val Loss: 10974.8408
Epoch [12/100] | Train Loss: 9933.1477 | Val Loss: 8626.4857
Epoch [13/100] | Train Loss: 7650.9789 | Val Loss: 6456.8694
Epoch [14/100] | Train Loss: 5618.1860 | Val Loss: 4572.4463
Epoch [15/100] | Train Loss: 3868.7481 | Val Loss: 3040.4223
Epoch [16/100] | Train Loss: 2506.9537 | Val Loss: 1913.0470
Epo